# Project 9: zware regen in Paramaribo, maand voor maand

**De vraag.** In welke maanden is een dag met zware regen in Paramaribo het waarschijnlijkst? Wat is in een willekeurige week van elke maand de kans op minstens één dag met zware regen?

**De data.** Dagelijkse regenval in millimeters voor Paramaribo, van 1 januari 1991 tot 31 december 2025, uit de archief-API van Open-Meteo met het model ERA5, gedownload op 2026-11-06. ERA5 is een heranalyse: een weermodel dat over voorbije jaren is doorgerekend en gecombineerd met waarnemingen van weerstations, vliegtuigen, boeien, radar en satellieten. Het schat de regen voor een rastercel van ongeveer 25 km breed, niet voor één regenmeter.

**Een dag met zware regen.** In dit project is een dag met zware regen een dag met minstens 10 mm regen. Dat is de drempel van R10mm, een index in de standaardlijst van klimaatindices die Climdex bijhoudt.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week leest de datacontroleur elke kans in de notitie en vraagt: een kans op wat, gegeven wat?

**Voor wie.** Een klein bouwbedrijf in Paramaribo dat betonstorten en dakwerk plant. Op vrijdag krijgt het bedrijf een regenkalender van één pagina: voor elke maand in hoeveel van de 10 weken er minstens één dag met zware regen was, met een korte notitie over hoeveel vertrouwen die kalender verdient.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, één staafdiagram, de regenkalender met de notitie van ongeveer 200 woorden, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of rechtstreeks bij Open-Meteo als het bestand er niet is. De tabel heeft twee kolommen: `date` en `rain_mm`, de regen die die dag viel in millimeters.

In [ ]:
import json, urllib.request
import pandas as pd

URL = ("https://archive-api.open-meteo.com/v1/archive?latitude=5.85&longitude=-55.2"
       "&start_date=1991-01-01&end_date=2025-12-31&daily=precipitation_sum"
       "&timezone=America%2FParamaribo&models=era5")
try:
    rain = pd.read_csv("https://santerre.ai/bigi-data/data/week09-paramaribo-rain.csv", parse_dates=["date"])
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(URL))["daily"]
    rain = pd.DataFrame({"date": pd.to_datetime(raw["time"]), "rain_mm": raw["precipitation_sum"]})
rain

## TODO 1: beschrijf de tabel

Sla het aantal rijen op in `n_rows` en het aantal ontbrekende waarden in `n_missing`. Druk de eerste en laatste datum af en het aandeel dagen zonder enige regen. De CHECK test ook dat er tussen de eerste en de laatste datum geen dag ontbreekt.

In [ ]:
# TODO 1
n_rows = None
n_missing = None

In [ ]:
# CHECK 1
assert n_rows is not None and n_missing is not None, "een waarde is nog leeg"
span = (rain["date"].max() - rain["date"].min()).days + 1
assert n_rows == len(rain) == span, "de tabel moet voor elke dag één rij hebben"
assert n_missing == int(rain["rain_mm"].isna().sum()) == 0
print("CHECK 1 geslaagd:", n_rows, "dagen, geen ontbrekend")

## TODO 2: markeer de dagen met zware regen

Voeg aan `rain` een kolom `heavy` toe die True is op dagen met minstens 10 mm regen. Sla daarna in `p_heavy` het aandeel van alle dagen op dat een dag met zware regen is. "Minstens 10 mm" omvat ook de dagen met precies 10 mm.

In [ ]:
# TODO 2
THRESHOLD_MM = 10
p_heavy = None

In [ ]:
# CHECK 2
assert p_heavy is not None, "p_heavy is nog leeg"
assert "heavy" in rain.columns, "voeg de kolom heavy toe aan rain"
assert (rain["heavy"] == (rain["rain_mm"] >= 10)).all(), "heavy moet True zijn op elke dag met minstens 10 mm"
assert abs(p_heavy - (rain["rain_mm"] >= 10).mean()) < 1e-9
print("CHECK 2 geslaagd:", int(rain["heavy"].sum()), "dagen met zware regen")

## TODO 3: de kans in elke maand

Sla in `p_month` de kans op een dag met zware regen in elke maand op, als een tabel met het maandnummer van 1 tot 12 als index. Elke waarde is het aandeel van de dagen van die maand, over alle 35 jaren, dat een dag met zware regen was. Sla de maand met de hoogste kans op in `top_month`.

In [ ]:
# TODO 3
p_month = None
top_month = None

In [ ]:
# CHECK 3
assert p_month is not None and top_month is not None, "een waarde is nog leeg"
expected = (rain["rain_mm"] >= 10).groupby(rain["date"].dt.month).mean()
assert len(p_month) == 12 and all(abs(p_month[k] - expected[k]) < 1e-9 for k in range(1, 13))
assert int(top_month) == int(expected.idxmax())
print("CHECK 3 geslaagd: de waarschijnlijkste maand is", int(top_month))

## TODO 4: maak een grafiek van de kans per maand

Maak een staafdiagram van `p_month`. Voeg een gestippelde horizontale lijn toe op de kans over het hele jaar uit TODO 2. Geef de grafiek een titel, benoem beide assen en noem de bron in de titel of een onderschrift.

In [ ]:
# TODO 4

## TODO 5: twee vragen die op elkaar lijken

In dit project zijn *de natte maanden* de maanden waarvan de kans uit TODO 3 boven de kans over het hele jaar ligt. Sla ze op in `wet_months`, een lijst met maandnummers.

Maak met `pd.crosstab` een kruistabel van "natte maand" (True of False) tegen "dag met zware regen" (True of False), met de marges. Sla daaruit twee getallen op:

- `p_heavy_given_wet`, de kans dat een dag in de natte maanden een dag met zware regen is;
- `p_wet_given_heavy`, het aandeel van de dagen met zware regen dat in de natte maanden valt.

In [ ]:
# TODO 5
wet_months = None
p_heavy_given_wet = None
p_wet_given_heavy = None

In [ ]:
# CHECK 5
assert all(v is not None for v in (wet_months, p_heavy_given_wet, p_wet_given_heavy)), "een waarde is nog leeg"
h = rain["rain_mm"] >= 10
mo = rain["date"].dt.month
wet = [k for k in range(1, 13) if h[mo == k].mean() > h.mean()]
assert sorted(int(k) for k in wet_months) == wet
in_wet = mo.isin(wet)
assert abs(p_heavy_given_wet - h[in_wet].mean()) < 1e-9, "p_heavy_given_wet: houd eerst de dagen in de natte maanden over"
assert abs(p_wet_given_heavy - in_wet[h].mean()) < 1e-9, "p_wet_given_heavy: houd eerst de dagen met zware regen over"
print("CHECK 5 geslaagd: de natte maanden zijn", wet)

## TODO 6: minstens één dag met zware regen in een week

Verdeel elke maand in vier weken: dag 1 tot 7, 8 tot 14, 15 tot 21 en 22 tot 28. Dag 29 tot 31 vallen weg, zodat elke week zeven dagen heeft. Maak een tabel `weekly` met één rij per maand (1 tot 12) en drie kolommen:

- `added`, zeven keer de dagkans uit TODO 3, de kortere weg die een AI-assistent vaak neemt;
- `independent`, `1 - (1 - p) ** 7`, de kans als de dagen onafhankelijk waren;
- `record`, het aandeel van al die weken in de 35 jaren met minstens één dag met zware regen.

Vergelijk de drie kolommen maand voor maand. Welke kolom beantwoordt de vraag, en in welke maanden gaan de andere twee de mist in?

In [ ]:
# TODO 6
weekly = None

In [ ]:
# CHECK 6
assert weekly is not None, "weekly is nog leeg"
d = rain[rain["date"].dt.day <= 28].assign(
    y=lambda t: t["date"].dt.year, mo=lambda t: t["date"].dt.month,
    wk=lambda t: (t["date"].dt.day - 1) // 7, h=lambda t: t["rain_mm"] >= 10)
weeks_any = d.groupby(["y", "mo", "wk"])["h"].any()
rec = weeks_any.groupby(level="mo").mean()
pm = (rain["rain_mm"] >= 10).groupby(rain["date"].dt.month).mean()
for k in range(1, 13):
    assert abs(weekly.loc[k, "added"] - 7 * pm[k]) < 1e-9, f"added klopt niet voor maand {k}"
    assert abs(weekly.loc[k, "independent"] - (1 - (1 - pm[k]) ** 7)) < 1e-9, f"independent klopt niet voor maand {k}"
    assert abs(weekly.loc[k, "record"] - rec[k]) < 1e-9, f"record klopt niet voor maand {k}"
print("CHECK 6 geslaagd:", int(weeks_any.groupby(level="mo").size().min()), "weken per maand in de gegevens")

## TODO 7: de AI-controle

Stel je AI-assistent drie vragen, zonder hem de tabel te laten zien, en schrijf elk antwoord in jullie AI-logboek.

1. Geef hem de dagkans die je in TODO 3 voor juli vond en vraag: *"In Paramaribo heeft een dag in juli deze kans op minstens 10 mm regen. Wat is de kans op minstens één zo'n dag in een willekeurige week van juli?"*
2. Stel dezelfde vraag voor januari, met de dagkans van januari.
3. Geef hem het aandeel van de dagen met zware regen dat in jullie natte maanden valt, uit TODO 5, en vraag: *"Wat is de kans dat een willekeurige dag in die maanden een dag met zware regen is?"*

Sla zijn drie antwoorden als getallen op in de cel hieronder. Sla `None` op voor het derde als hij zegt dat de vraag niet alleen met dat aandeel te beantwoorden is. De CHECK drukt elk antwoord af naast de getallen van het notebook. Zeg in jullie AI-logboek bij welke kolom elk antwoord past, en of de assistent zei wat hij aannam.

In [ ]:
# TODO 7
ai_july = None      # het antwoord van de AI voor juli
ai_january = None   # het antwoord van de AI voor januari
ai_wet_day = None   # het antwoord van de AI op vraag 3, of None

In [ ]:
# CHECK 7
assert ai_july is not None and ai_january is not None, "schrijf de twee weekantwoorden van de AI op"
for name, k, ai in [("juli", 7, ai_july), ("januari", 1, ai_january)]:
    print(name, "| AI:", round(ai, 3), "| added:", round(weekly.loc[k, "added"], 3),
          "| independent:", round(weekly.loc[k, "independent"], 3), "| record:", round(weekly.loc[k, "record"], 3))
print("een dag in de natte maanden | AI:", ai_wet_day, "| P(zwaar gegeven nat):", round(p_heavy_given_wet, 3),
      "| P(nat gegeven zwaar):", round(p_wet_given_heavy, 3))
print("CHECK 7 geslaagd: de antwoorden van de AI staan naast de data")

## TODO 8: verandert de drempel het antwoord?

Herhaal TODO 3 en TODO 5 met een drempel van 20 mm, de drempel van de index R20mm. Sla de maand met de hoogste kans op in `top_month_20` en de natte maanden in `wet_months_20`. Beslis daarna of het antwoord op de projectvraag afhangt van waar je de drempel legt.

In [ ]:
# TODO 8
top_month_20 = None
wet_months_20 = None

In [ ]:
# CHECK 8
assert top_month_20 is not None and wet_months_20 is not None, "een waarde is nog leeg"
h20 = rain["rain_mm"] >= 20
p20 = h20.groupby(rain["date"].dt.month).mean()
assert int(top_month_20) == int(p20.idxmax())
assert sorted(int(k) for k in wet_months_20) == [k for k in range(1, 13) if p20[k] > h20.mean()]
print("CHECK 8 geslaagd: bij 20 mm is de waarschijnlijkste maand", int(top_month_20))

## TODO 9: de regenkalender

Maak `calendar`, de tabel die het bouwbedrijf aan de muur hangt. De tabel heeft één rij per maand, van januari tot december, en twee kolommen: `month`, de naam van de maand, en `weeks_out_of_10`, het aantal weken van de 10 met minstens één dag met zware regen. Haal dat uit de kolom `record` van TODO 6 en rond het af op een heel getal. Druk de kalender af zonder de index, zodat hij leest als een kalender en niet als code.

In [ ]:
# TODO 9
MONTH_NAMES = ["januari", "februari", "maart", "april", "mei", "juni",
               "juli", "augustus", "september", "oktober", "november", "december"]
calendar = None

In [ ]:
# CHECK 9
assert calendar is not None, "calendar is nog leeg"
assert list(calendar.columns) == ["month", "weeks_out_of_10"], "gebruik de kolommen month en weeks_out_of_10"
assert len(calendar) == 12
assert [int(v) for v in calendar["weeks_out_of_10"]] == [int(v) for v in (weekly["record"] * 10).round()]
print("CHECK 9 geslaagd: de kalender heeft", len(calendar), "maanden")

## TODO 10: de notitie bij de kalender

Schrijf in een tekstcel hieronder ongeveer 200 woorden voor het bouwbedrijf, om onder de kalender af te drukken. Zeg in welke maanden de kans op een dag met zware regen het hoogst is. Lees twee rijen van de kalender in gewone woorden, bijvoorbeeld "in ongeveer N van de 10 weken viel er op minstens één dag zware regen". Noem de bron, en zeg twee dingen die de data het bedrijf niet kan vertellen.